In [6]:
import os
os.chdir("/home/ruslan/thesis/cl")

In [7]:
import torch
from memories.memory_stream import StreamManager

from moviad.datasets.bmad.bmad_dataset import BMAD

from memories.replay_strategy import ReplayModel
from trainers.models import STFPMModel


continual_dataset = StreamManager(BMAD, task_type="segmentation", root_dir="/mnt/disk1/ruslan_nuriev/bmad")

# Create model and strategy
replay_strategy = ReplayModel(
    model_conf={'stfpm': STFPMModel("cuda:0", 'resnet18', ['layer1', 'layer2', 'layer3'])},
    buffer_size=1000
)


(train) Task 0 (liver): 1542 samples
(train) Task 1 (chest): 8000 samples
(train) Task 2 (histopathology): 5088 samples
(train) Task 3 (brain): 7500 samples
(train) Task 4 (retinaoct): 26315 samples
(train) Task 5 (retinaresc): 4297 samples
(test) Task 0 (liver): 1493 samples
(test) Task 1 (chest): 17194 samples
(test) Task 2 (histopathology): 1997 samples
(test) Task 3 (brain): 3715 samples
(test) Task 4 (retinaoct): 968 samples
(test) Task 5 (retinaresc): 1805 samples


In [ ]:
from trainers.continual_trainer import ContinualTrainer

trainer = ContinualTrainer(
    strategy=replay_strategy
    )


In [ ]:
trainer.strategy.replay_buffer.buffer

In [ ]:
# Train
history = trainer.train(
    continual_dataset,
    epochs_per_task=1
)

# Save final model

In [ ]:
# from moviad.utilities.custom_feature_extractor_trimmed import CustomFeatureExtractor
# from moviad.datasets.mvtec.mvtec_dataset import MVTecDataset
# from moviad.models.stfpm.stfpm import STFPM
# import torch

# dataset_path = "/mnt/disk1/manuel_barusco/CL_VAD/adcl_paper/data/mvtec"

# ad_layers = ["layer1", "layer2", "layer3"]
# student = CustomFeatureExtractor('wide_resnet50_2', ad_layers, device="cuda:0") #128 - 512
# teacher = CustomFeatureExtractor('wide_resnet50_2', ad_layers, device="cuda:0")

# # test_dataset = MVTecDataset("segmentation", dataset_path, "hazelnut", "test")
# # test_dataset.load_dataset()
# # test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=5, shuffle=False)
# # img = next(iter(test_dataloader))
# # # print(img)


# model = STFPM(teacher, student)
# # for param_tensor in model.student.model.state_dict():
# #     print(param_tensor, "\t", model.state_dict()[param_tensor].size())
# #     print("not")
# # model.load_state_dict(torch.load("/home/ruslan/thesis/checkpoints/model_weights.pth", weights_only=True))
# # model.eval()
# # for param_tensor in model.state_dict():
# #     print(param_tensor, "\t", model.state_dict()[param_tensor].size())
# #     print("hot")

# print(model.model)


In [ ]:
from moviad.models.stfpm.stfpm import STFPM
from moviad.utilities.custom_feature_extractor_trimmed import CustomFeatureExtractor
import torch.nn.functional as F
from moviad.trainers.trainer_stfpm import TrainerSTFPM
import numpy as np

teacher = CustomFeatureExtractor("resnet18", ["layer1", "layer2", "layer3"], "cuda:0")
student = CustomFeatureExtractor("resnet18", ["layer1", "layer2", "layer3"], "cuda:0", False)
stfpm_model = STFPM(teacher, student)
stfpm_model.train()

for task in range(continual_dataset.num_categories):
    train_loader, test_loader = continual_dataset.get_current_task_loaders()
    for epoch in range(1):
        epoch_losses = []
        for batch in train_loader:
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            
            teacher_features, student_features = stfpm_model(images.to('cuda:0'))
            loss = 0
            for i in range(len(teacher_features)):
                teacher_features[i] = F.normalize(teacher_features[i], dim=1)
                student_features[i] = F.normalize(student_features[i], dim=1)
                loss += TrainerSTFPM.stfpm_loss(teacher_features[i], student_features[i])
            

            epoch_losses.append(loss.cpu().detach())
        
        avg_loss = np.mean(epoch_losses)
        print(f"  Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    if not continual_dataset.to_next_task():
                break

  Epoch 1, Loss: 3.6893
Moved to task 1: chest
Moved to task 1: chest
  Epoch 1, Loss: 3.5983
Moved to task 2: histopathology
Moved to task 2: histopathology


KeyboardInterrupt: 

In [15]:
import torch

torch.cuda.empty_cache()

In [10]:
del train_loader

In [11]:
del test_loader

In [14]:
%reset